# STAC Catalog Builder — Groundwater recharge in Africa (two COGs)

This notebook generates a STAC catalog for the preprocessed groundwater recharge COGs and uploads it to Google Cloud.
It follows `17_seabed_litho.ipynb` (COG assets, `CoCliCoCOGLayout`, cloud upload, GeoServer WMS), with **one collection and two items**, one per model variant (`RF_recharge_01`, `RF_RK_recharge_01`). Each item carries the model in `recharge:model` and a `visual` WMS asset.

## Pipeline overview

| Step | Purpose |
|------|---------|
| 1. Configure paths and options | Input COGs + metadata on `P:`, cloud targets (`gca-stac-7`) |
| 2. Define STAC helper functions | Collection, item and asset builders |
| 3. Create the STAC collection | From `metadata_groundwater_recharge.json` |
| 4. Build the STAC items | One item per COG |
| 5. Save STAC catalog locally | `STAC/data/current/groundwater_recharge/` + root `catalog.json` |
| 6. Upload COGs to Google Cloud | `gs://gca-data-public/gca/groundwater_recharge/*.tif` |
| 7. Upload STAC catalog to Google Cloud | `gs://gca-data-public/gca/gca-stac-7/` |
| 8. Publish the layers to GeoServer WMS | One cell, URL/credentials from the git-ignored `.env` |
| 9. Verify the cloud copy | Public HTTPS hrefs reachable and same size as the local COGs |

**Prerequisite:** run `notebooks/20_groundwater_recharge.ipynb` first (via `docker/run_notebook_with_p_drive.ps1`) to produce `cog/*.tif` and `metadata_groundwater_recharge.json` in `P:\...\Data_Vivian\stac\groundwater_recharge`.

Steps 6-7 are guarded by `RUN_UPLOAD`, step 8 by `RUN_GEOSERVER_PUBLISH`.

In [1]:
import datetime
import json
import os
from pathlib import Path
from posixpath import join as urljoin

import pystac
import rasterio
import requests
import rioxarray
import shapely
import xarray as xr
from pystac.extensions import eo, raster
from pystac.stac_io import DefaultStacIO

from coclicodata.coclico_stac.layouts import CoCliCoCOGLayout
from coclicodata.etl.cloud_utils import load_google_credentials

## 1) Configure paths and options

In [2]:
stac_data_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))
stac_dir = stac_data_repo / "current"

data_dir = Path(r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge")
cog_dir = data_dir / "cog"
metadata_path = data_dir / "metadata_groundwater_recharge.json"
google_cred_path = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\google_credentials.json"
)

RUN_UPLOAD = True  # set False to only build the STAC locally
gcs_protocol = "https://storage.googleapis.com"
gcs_project = "GCA - 11210264"
bucket_name = "gca-data-public"
bucket_proj = "gca"
stac_cloud_name = "gca-stac-7"

if not metadata_path.is_file():
    raise FileNotFoundError(f"{metadata_path} not found. Run notebooks/20_groundwater_recharge.ipynb first.")

with open(metadata_path, encoding="utf-8") as f:
    metadata = json.load(f)

REQUIRED_METADATA_KEYS = [
    "SPATIAL_RESOLUTION", "CRS", "ITEM_BBOX_CRS", "NODATA", "DATA_TYPE", "COLLECTION_ID", "SPATIAL_EXTENT",
    "TEMPORAL_EXTENT", "TITLE", "DESCRIPTION", "LICENSE", "PROVIDERS", "KEYWORDS", "UNITS", "ITEMS",
]
missing_keys = [key for key in REQUIRED_METADATA_KEYS if key not in metadata]
if missing_keys:
    raise KeyError(f"Missing required metadata keys in {metadata_path}: {', '.join(missing_keys)}")

cog_paths = [cog_dir / f"{name}.tif" for name in metadata["ITEMS"]]
missing = [str(p) for p in cog_paths if not p.is_file()]
if missing:
    raise FileNotFoundError(f"COGs not found: {missing}. Run notebooks/20_groundwater_recharge.ipynb first.")

collection_id = metadata["COLLECTION_ID"]
proj_name = collection_id
href_prefix = urljoin(gcs_protocol, bucket_name, bucket_proj, proj_name)

# GeoServer WMS (step 8): workspace and visual asset name
wms_dataset = metadata.get("WMS_DATASET") or collection_id
wms_base_url = "https://international-delta-platform.avi.directory.intra/geoserver/wms"
geoserver_gcs_mount_prefix = "/opt/gca-data-public/gca"  # where the GCS bucket is mounted on the GeoServer host

print("COG inputs:", *cog_paths, sep="\n  ")
print("STAC output:", stac_dir / collection_id)
print("Collection id:", collection_id)
print("Spatial extent:", metadata["SPATIAL_EXTENT"])
print("Temporal extent:", metadata["TEMPORAL_EXTENT"])
print("Cloud COG hrefs:", *[urljoin(href_prefix, p.name) for p in cog_paths], sep="\n  ")

COG inputs:
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge\cog\RF_recharge_01.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge\cog\RF_RK_recharge_01.tif
STAC output: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\groundwater_recharge
Collection id: groundwater_recharge
Spatial extent: [-25.375, -34.825, 63.475, 37.325]
Temporal extent: ['1970-01-01T00:00:00Z', '2020-12-31T23:59:59Z']
Cloud COG hrefs:
  https://storage.googleapis.com/gca-data-public/gca/groundwater_recharge/RF_recharge_01.tif
  https://storage.googleapis.com/gca-data-public/gca/groundwater_recharge/RF_RK_recharge_01.tif


## 2) Define STAC helper functions

In [3]:
DATA_TYPE_MAP = {
    "float32": raster.DataType.FLOAT32,
    "float64": raster.DataType.FLOAT64,
    "int8": raster.DataType.INT8,
    "int16": raster.DataType.INT16,
    "int32": raster.DataType.INT32,
    "uint8": raster.DataType.UINT8,
    "uint16": raster.DataType.UINT16,
}


def get_raster_data_type(metadata: dict):
    data_type = metadata["DATA_TYPE"].lower()
    if data_type not in DATA_TYPE_MAP:
        raise ValueError(f"Unsupported DATA_TYPE in metadata: {metadata['DATA_TYPE']}")
    return DATA_TYPE_MAP[data_type]


def parse_license(metadata: dict) -> str:
    license_ = metadata["LICENSE"]
    if "Creative Commons" in license_ and "4.0" in license_:
        suffix = "-SA" if "ShareAlike" in license_ else ""
        return f"CC-BY{suffix}-4.0"
    return license_


def parse_datetime(value: str) -> datetime.datetime:
    return datetime.datetime.fromisoformat(value.replace("Z", "+00:00"))


def create_collection(metadata: dict, collection_id: str) -> pystac.Collection:
    providers = [
        pystac.Provider(
            name=metadata["PROVIDERS"]["name"],
            roles=[pystac.provider.ProviderRole.PRODUCER, pystac.provider.ProviderRole.LICENSOR],
            url=metadata["PROVIDERS"]["url"],
        ),
        pystac.Provider(
            name="Deltares",
            roles=[pystac.provider.ProviderRole.PROCESSOR, pystac.provider.ProviderRole.HOST],
            url="https://deltares.nl",
        ),
    ]

    start, end = (parse_datetime(v) for v in metadata["TEMPORAL_EXTENT"])
    extent = pystac.Extent(
        pystac.SpatialExtent([metadata["SPATIAL_EXTENT"]]),
        pystac.TemporalExtent([[start, end]]),
    )

    collection = pystac.Collection(
        id=collection_id,
        title=metadata["TITLE"],
        description=metadata["DESCRIPTION"],
        license=parse_license(metadata),
        providers=providers,
        extent=extent,
        catalog_type=pystac.CatalogType.RELATIVE_PUBLISHED,
    )
    collection.keywords = metadata["KEYWORDS"]

    pystac.extensions.item_assets.ItemAssetsExtension.add_to(collection)
    collection.extra_fields["item_assets"] = {
        "data": {
            "type": pystac.MediaType.COG,
            "title": metadata["TITLE"],
            "roles": ["data"],
            "description": metadata["SHORT_DESCRIPTION"],
            "xarray:storage_options": {"token": "google_default"},
        }
    }
    collection.extra_fields["deltares:units"] = metadata["UNITS"]

    pystac.extensions.scientific.ScientificExtension.add_to(collection)
    collection.extra_fields["sci:citation"] = metadata["CITATION"]
    if metadata.get("DOI"):
        collection.extra_fields["sci:doi"] = metadata["DOI"].removeprefix("https://doi.org/")
    return collection


def create_item(block, item_id: str, metadata: dict) -> pystac.Item:
    bbox = list(metadata["SPATIAL_EXTENT"])
    geometry = shapely.geometry.mapping(shapely.geometry.box(*bbox))

    # Period average (1970-2020): the item covers the whole period
    start, end = (parse_datetime(v) for v in metadata["TEMPORAL_EXTENT"])
    info = metadata["ITEMS"][item_id]
    item = pystac.Item(
        id=item_id, geometry=geometry, bbox=bbox, datetime=start,
        properties={"title": info["TITLE"], "description": info["DESCRIPTION"]},
    )
    item.common_metadata.start_datetime = start
    item.common_metadata.end_datetime = end
    item.common_metadata.created = datetime.datetime.now(datetime.timezone.utc)

    ext = pystac.extensions.projection.ProjectionExtension.ext(item, add_if_missing=True)
    ext.bbox = block.rio.bounds()
    ext.shape = tuple(v for k, v in block.sizes.items() if k in ["y", "x"])
    ext.epsg = block.rio.crs.to_epsg()
    ext.geometry = shapely.geometry.mapping(shapely.geometry.box(*ext.bbox))
    ext.transform = list(block.rio.transform())[:6]
    ext.add_to(item)

    item.properties["deltares:item_key"] = item_id
    item.properties["recharge:model"] = info["MODEL"]
    return item


def create_asset(item, asset_title: str, asset_href: str, nodata, resolution, data_type, description: str, nbytes=None):
    asset = pystac.Asset(href=asset_href, media_type=pystac.MediaType.COG, title=asset_title, roles=["data"])
    item.add_asset("data", asset)
    pystac.extensions.file.FileExtension.ext(asset, add_if_missing=True)
    if nbytes:
        asset.extra_fields["file:size"] = nbytes
    raster.RasterExtension.ext(asset, add_if_missing=True).bands = [
        raster.RasterBand.create(nodata=nodata, spatial_resolution=resolution, data_type=data_type, unit=metadata["UNITS"])
    ]
    eo.EOExtension.ext(asset, add_if_missing=True).bands = [
        eo.Band.create(name=asset_title, description=description)
    ]
    return item


def create_wms_asset(item, asset_title: str, asset_href: str):
    asset = pystac.Asset(href=asset_href, media_type="application/png", title=asset_title, description="OGS WMS url", roles=["visual"])
    item.add_asset("visual", asset)
    return item


def process_block(file_path: Path, metadata: dict, storage_prefix: str = "") -> pystac.Item:
    block = xr.open_dataset(file_path, engine="rasterio", mask_and_scale=False)
    item_id = file_path.stem
    item = create_item(block, item_id=item_id, metadata=metadata)

    create_asset(
        item,
        asset_title=item_id,
        asset_href=urljoin(storage_prefix, file_path.name),
        nodata=metadata["NODATA"],
        resolution=metadata["SPATIAL_RESOLUTION"],
        data_type=get_raster_data_type(metadata),
        description=metadata["ITEMS"][item_id]["DESCRIPTION"],
        nbytes=os.path.getsize(file_path),
    )
    create_wms_asset(item, asset_title=item_id, asset_href=f"{wms_base_url}/{wms_dataset}")
    return item

## 3) Create the STAC collection

In [4]:
collection = create_collection(metadata, collection_id)

print(f"Collection ready: {collection.id}")
print(f"Title: {metadata['TITLE']}")
print(f"Provider: {metadata['PROVIDERS']['name']}")
print(f"License: {collection.license}")
print(f"Spatial extent: {metadata['SPATIAL_EXTENT']}")
print(f"Temporal extent: {metadata['TEMPORAL_EXTENT']}")

Collection ready: groundwater_recharge
Title: Groundwater Recharge in Africa from Ground Based Measurements
Provider: Figshare
License: CC-BY-4.0
Spatial extent: [-25.375, -34.825, 63.475, 37.325]
Temporal extent: ['1970-01-01T00:00:00Z', '2020-12-31T23:59:59Z']


## 4) Build the STAC items

One item per COG. Item id `RF_recharge_01` is turned into `items/RF_recharge_01.json` by `CoCliCoCOGLayout`.

In [5]:
items = []
for cog_path in cog_paths:
    item = process_block(cog_path, metadata, storage_prefix=href_prefix)
    item.set_self_href(str(stac_dir / collection_id / "items" / f"{cog_path.stem}.json"))
    collection.add_item(item)
    items.append(item)

    print(f"Item id: {item.id} | model: {item.properties['recharge:model']}")
    print(f"  bbox: {item.bbox}")
    print(f"  data asset href: {item.assets['data'].href}")
    print(f"  WMS asset href: {item.assets['visual'].href}")
    print(f"  raster bands: {item.assets['data'].extra_fields.get('raster:bands')}")

Item id: RF_recharge_01 | model: RF
  bbox: [-25.375, -34.825, 63.475, 37.325]
  data asset href: https://storage.googleapis.com/gca-data-public/gca/groundwater_recharge/RF_recharge_01.tif
  WMS asset href: https://international-delta-platform.avi.directory.intra/geoserver/wms/groundwater_recharge
  raster bands: [{'nodata': -9999.0, 'data_type': 'float32', 'spatial_resolution': 0.1, 'unit': 'mm/year'}]
Item id: RF_RK_recharge_01 | model: RF_RK
  bbox: [-25.375, -34.825, 63.475, 37.325]
  data asset href: https://storage.googleapis.com/gca-data-public/gca/groundwater_recharge/RF_RK_recharge_01.tif
  WMS asset href: https://international-delta-platform.avi.directory.intra/geoserver/wms/groundwater_recharge
  raster bands: [{'nodata': -9999.0, 'data_type': 'float32', 'spatial_resolution': 0.1, 'unit': 'mm/year'}]


## 5) Save STAC catalog locally

Writes `STAC/data/current/groundwater_recharge/` and adds the collection to the root `catalog.json`.

In [6]:
stac_io = DefaultStacIO()
layout = CoCliCoCOGLayout()

(stac_dir / collection_id / "items").mkdir(parents=True, exist_ok=True)

collection.update_extent_from_items()

catalog = pystac.Catalog.from_file(str(stac_dir / "catalog.json"))
if catalog.get_child(collection.id):
    catalog.remove_child(collection.id)
    print(f"Removed existing child: {collection.id}")

catalog.add_child(collection)
collection.normalize_hrefs(str(stac_dir / collection_id), strategy=layout)
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED, dest_href=str(stac_dir), stac_io=stac_io)

collection.validate_all()
catalog.validate_all()

expected = [stac_dir / collection_id / "collection.json"] + [stac_dir / collection_id / "items" / f"{i.id}.json" for i in items]
missing = [str(p) for p in expected if not p.is_file()]
if missing:
    raise FileNotFoundError(f"STAC outputs incomplete after save: {missing}")
print(f"STAC saved to: {stac_dir}")

Removed existing child: groundwater_recharge
STAC saved to: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current


## 6) Upload COGs to Google Cloud (optional)

Uploads the COGs to `gs://gca-data-public/gca/groundwater_recharge/`, matching the hrefs in the STAC items. Existing remote copies are replaced.

In [7]:
if RUN_UPLOAD:
    import gcsfs

    load_google_credentials(google_token_fp=google_cred_path)
    fs = gcsfs.GCSFileSystem(gcs_project, token=os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

    for cog_path in cog_paths:
        target = urljoin(bucket_name, bucket_proj, proj_name, cog_path.name)
        print(f"Uploading {cog_path.name} -> gs://{target} ...")
        if fs.exists(target):
            print("  Removing existing remote COG")
            fs.rm(target)
        fs.put(str(cog_path), target)
    print("Done")
else:
    print("RUN_UPLOAD is False, skipping COG upload.")

C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:339: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:352: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Google Application Credentials load into environment.
Uploading RF_recharge_01.tif -> gs://gca-data-public/gca/groundwater_recharge/RF_recharge_01.tif ...
  Removing existing remote COG
Uploading RF_RK_recharge_01.tif -> gs://gca-data-public/gca/groundwater_recharge/RF_RK_recharge_01.tif ...
  Removing existing remote COG
Done


## 7) Upload STAC catalog to Google Cloud (optional)

Replaces only this collection plus the root `catalog.json` at `gs://gca-data-public/gca/gca-stac-7/`.

In [8]:
if RUN_UPLOAD:
    local_collection_dir = stac_dir / collection_id
    remote_collection = urljoin(bucket_name, bucket_proj, stac_cloud_name, collection_id)
    remote_catalog = urljoin(bucket_name, bucket_proj, stac_cloud_name, "catalog.json")

    if fs.exists(remote_collection):
        print("Removing existing remote collection...")
        fs.rm(remote_collection, recursive=True)

    print("Uploading collection...")
    fs.put(str(local_collection_dir), remote_collection, recursive=True)
    print("Uploading root catalog.json...")
    fs.put(str(stac_dir / "catalog.json"), remote_catalog)

    print(f"Uploaded collection: gs://{remote_collection}/")
    print(f"Uploaded catalog: gs://{remote_catalog}")
else:
    print("RUN_UPLOAD is False, skipping STAC catalog upload.")

Removing existing remote collection...
Uploading collection...
Uploading root catalog.json...
Uploaded collection: gs://gca-data-public/gca/gca-stac-7/groundwater_recharge/
Uploaded catalog: gs://gca-data-public/gca/gca-stac-7/catalog.json


## 8) Publish the layers to GeoServer WMS (optional)

One cell: registers each COG uploaded in step 6 as an external coverage store in GeoServer (workspace = `WMS_DATASET`, one layer per COG), as done in `IDP-workbench/WMS/geoserver_workspace_s.py` and step 8 of `17_seabed_litho.ipynb`.

- URL and credentials come from the git-ignored `global-coastal-atlas/.env` (copy `.env.example`): `GEOSERVER_URL`, `GEOSERVER_USERNAME`, `GEOSERVER_PASSWORD`, optional `GEOSERVER_VERIFY_SSL` (default `false`, self-signed certificate).
- Run it after step 6: GeoServer reads the COGs from the bucket mount (`/opt/gca-data-public/gca/groundwater_recharge/<name>.tif`).
- `GEOSERVER_STYLE`: name of an existing GeoServer style for a continuous colour ramp in mm/year. `None` keeps the GeoServer default style (grey ramp).

In [9]:
RUN_GEOSERVER_PUBLISH = True
GEOSERVER_STYLE = None  # e.g. "groundwater_recharge" once an SLD with a colour ramp exists

if RUN_GEOSERVER_PUBLISH:
    import time

    import urllib3
    from dotenv import load_dotenv
    from geo.Geoserver import Geoserver

    env_path = stac_data_repo.parent.parent / ".env"  # global-coastal-atlas/.env (git-ignored)
    if not env_path.is_file():
        raise FileNotFoundError(f"{env_path} not found. Copy .env.example to .env and fill in the GeoServer URL and credentials.")
    load_dotenv(env_path, override=True)
    geoserver_url = os.environ["GEOSERVER_URL"].rstrip("/")
    geoserver_user = os.environ["GEOSERVER_USERNAME"]
    geoserver_pwd = os.environ["GEOSERVER_PASSWORD"]
    verify_ssl = os.environ.get("GEOSERVER_VERIFY_SSL", "false").lower() == "true"

    if not verify_ssl:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        if not getattr(requests.Session.request, "_no_verify", False):  # patch once, also when the cell is re-run
            _request = requests.Session.request

            def _request_no_verify(self, method, url, **kwargs):
                kwargs.setdefault("verify", False)
                return _request(self, method, url, **kwargs)

            _request_no_verify._no_verify = True
            requests.Session.request = _request_no_verify

    geo = Geoserver(geoserver_url, username=geoserver_user, password=geoserver_pwd)
    workspace = wms_dataset

    try:
        geo.create_workspace(workspace=workspace)
        print(f"Workspace '{workspace}' created")
    except Exception as exc:
        if "already exists" in str(exc).lower() or "409" in str(exc):
            print(f"Workspace '{workspace}' already exists")
        else:
            raise

    for cog_path in cog_paths:
        layer_name = cog_path.stem
        server_tif_path = urljoin(geoserver_gcs_mount_prefix, proj_name, cog_path.name)

        # Replace an existing layer/store with the same name
        try:
            geo.delete_layer(workspace=workspace, layer_name=layer_name)
            geo.delete_coveragestore(workspace=workspace, store_name=layer_name)
            print(f"Old layer/store removed: {layer_name}")
        except Exception:
            pass

        geo.create_coveragestore(layer_name=layer_name, path=server_tif_path, workspace=workspace, method="external")
        print(f"Layer created: {workspace}:{layer_name} <- {server_tif_path}")

        if GEOSERVER_STYLE:
            resp = requests.put(
                f"{geoserver_url}/rest/layers/{workspace}:{layer_name}.json",
                auth=(geoserver_user, geoserver_pwd),
                json={"layer": {"defaultStyle": {"name": GEOSERVER_STYLE}}},
                verify=verify_ssl,
            )
            print(f"Style '{GEOSERVER_STYLE}':", "set" if resp.status_code in (200, 201) else f"FAILED {resp.status_code} {resp.text}")
        time.sleep(1)

    print(f"WMS: {geoserver_url}/{workspace}/wms?SERVICE=WMS&VERSION=1.1.1&REQUEST=GetCapabilities")
else:
    print("RUN_GEOSERVER_PUBLISH is False, skipping GeoServer publishing.")

Workspace 'groundwater_recharge' already exists
Layer created: groundwater_recharge:RF_recharge_01 <- /opt/gca-data-public/gca/groundwater_recharge/RF_recharge_01.tif
Layer created: groundwater_recharge:RF_RK_recharge_01 <- /opt/gca-data-public/gca/groundwater_recharge/RF_RK_recharge_01.tif
WMS: https://international-delta-platform.avi.directory.intra/geoserver/groundwater_recharge/wms?SERVICE=WMS&VERSION=1.1.1&REQUEST=GetCapabilities


## 9) Verify the cloud copy

Checks the public HTTPS hrefs of the COGs and the catalog, and that the COG sizes match the local files. For a pixel-level comparison (raw vs `P:` stac vs cloud), use `exploration/20_groundwater_recharge_check.ipynb`.

In [10]:
for item, cog_path in zip(items, cog_paths):
    cog_url = item.assets["data"].href
    r = requests.head(cog_url, allow_redirects=True, timeout=30)
    print(cog_url, "->", r.status_code)
    remote_size = int(r.headers.get("Content-Length", -1))
    print("  Remote size:", remote_size, "| local size:", cog_path.stat().st_size)
    assert r.status_code == 200, "COG not reachable (is the bucket public / was the upload run?)"
    assert remote_size == cog_path.stat().st_size, "Remote COG size differs from the local COG"

catalog_url = urljoin(gcs_protocol, bucket_name, bucket_proj, stac_cloud_name, "catalog.json")
r = requests.get(catalog_url, params={"nocache": datetime.datetime.now().timestamp()}, timeout=30)  # public objects are cached for 1 h
print(catalog_url, "->", r.status_code)
assert r.status_code == 200
assert f"./{collection_id}/collection.json" in r.text, "collection not linked in the cloud catalog"
print("OK: cloud COGs and catalog are in place.")

https://storage.googleapis.com/gca-data-public/gca/groundwater_recharge/RF_recharge_01.tif -> 200
  Remote size: 887597 | local size: 887597
https://storage.googleapis.com/gca-data-public/gca/groundwater_recharge/RF_RK_recharge_01.tif -> 200
  Remote size: 898941 | local size: 898941
https://storage.googleapis.com/gca-data-public/gca/gca-stac-7/catalog.json -> 200
OK: cloud COGs and catalog are in place.
